# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ShamKottish/FlyRankML/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [3]:
%pip -q install duckdb huggingface_hub

import os
import getpass
import duckdb
import pandas as pd
import numpy as np

# Get Hugging Face token safely
HF_TOKEN = os.environ.get("HF_TOKEN")

if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get("HF_TOKEN")
    except Exception:
        pass

HF_TOKEN = HF_TOKEN or getpass.getpass(
    "Paste your Hugging Face READ token: "
)

con = duckdb.connect()

con.execute(
    f"CREATE OR REPLACE SECRET hf "
    f"(TYPE huggingface, TOKEN '{HF_TOKEN}')"
)

REL = "hf://datasets/FlyRank/internship-warehouse"

FACT_MAR = (
    f"read_parquet('{REL}/"
    "fact_content_daily_performance/month=2026-03/*.parquet')"
)

FACT_APR = (
    f"read_parquet('{REL}/"
    "fact_content_daily_performance/month=2026-04/*.parquet')"
)

DIM_CLIENTS = f"read_parquet('{REL}/dim_clients.parquet')"

print("Connected to FlyRank warehouse.")

Paste your Hugging Face READ token: ··········
Connected to FlyRank warehouse.


## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*


**Unit of analysis:** one pseudonymized content item for one pseudonymized client.

For this provisional CTR opportunity contract, I use two separate time windows:

- **Feature window:** March 1–31, 2026
- **Outcome window:** April 1–30, 2026

March measurements are information available before the outcome period and may be used as features. April measurements are reserved for evaluating the later CTR opportunity outcome and must never be used as March features.

The final modeling table will contain one row per `client_hash_id + content_hash_id`. I will require enough search exposure to reduce low-volume noise.

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Verify the source date windows and basic row counts.

window_check = con.sql(f"""
    SELECT
        'feature_March' AS window_name,
        COUNT(*) AS rows,
        MIN(report_date) AS min_date,
        MAX(report_date) AS max_date
    FROM {FACT_MAR}

    UNION ALL

    SELECT
        'outcome_April',
        COUNT(*),
        MIN(report_date),
        MAX(report_date)
    FROM {FACT_APR}
""").df()

display(window_check)

assert str(window_check.loc[0, "min_date"])[:10] == "2026-03-01"
assert str(window_check.loc[0, "max_date"])[:10] == "2026-03-31"

assert str(window_check.loc[1, "min_date"])[:10] == "2026-04-01"
assert str(window_check.loc[1, "max_date"])[:10] == "2026-04-30"

print("✓ Feature and outcome windows do not overlap.")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,window_name,rows,min_date,max_date
0,feature_March,9841378,2026-03-01,2026-03-31
1,outcome_April,10424730,2026-04-01,2026-04-30


✓ Feature and outcome windows do not overlap.


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

### Features

The features come only from the March 2026 feature window:

- `feature_impressions` — total observed GSC impressions
- `feature_clicks` — total observed GSC clicks
- `feature_ctr` — clicks divided by impressions
- `feature_avg_position` — average observed search position

These fields describe search visibility before the April outcome period.

### Label / proxy

The April measurements are reserved for constructing the later CTR opportunity proxy:

- `outcome_impressions`
- `outcome_clicks`
- `outcome_ctr`
- `outcome_avg_position`
- `outcome_position_tier`
- `outcome_tier_median_ctr`
- `ctr_gap_pp`
- `opportunity_proxy`

The proxy identifies pages with enough April impressions whose CTR is meaningfully below comparable pages in the same position tier. It is a rule-defined decision-support proxy, not causal ground truth.

### Context

- `client_hash_id`
- `content_hash_id`
- `gsc_data_start`

These fields are used for grouping, joins, history checks, and validation. The identifiers will never be model features.

### Excluded

- April outcome measurements are excluded from March features because using them would leak future information.
- `client_hash_id` and `content_hash_id` are excluded as predictive features because they are identifiers.
- Raw client names, domains, URLs, titles, and private queries are excluded and are not present in the pseudonymized release.
- Product-generated decision flags or scores are excluded because the model should learn from observable measurements rather than copy an existing product decision.
- Rows without valid GSC availability are excluded from search-performance calculations.

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Check that the fields used in this contract actually exist.

schema = con.sql(f"""
    DESCRIBE SELECT * FROM {FACT_MAR}
""").df()

available_columns = set(schema["column_name"])

required = {
    "report_date",
    "client_hash_id",
    "content_hash_id",
    "gsc_impressions",
    "gsc_clicks",
    "gsc_avg_position",
}

missing = required - available_columns

assert not missing, f"Missing expected fields: {sorted(missing)}"

print("✓ All fields required for the CTR contract exist.")
print("\nFields used:")
for col in sorted(required):
    print("-", col)

✓ All fields required for the CTR contract exist.

Fields used:
- client_hash_id
- content_hash_id
- gsc_avg_position
- gsc_clicks
- gsc_impressions
- report_date


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

I verify four parts of the contract below:

1. the raw warehouse grain,
2. the one-row-per-content modeling grain after aggregation,
3. the number of rows with missing or unusable search measurements,
4. the separation between the March feature window and April outcome window.

The final analysis table keeps only content items with enough observed search exposure to make CTR comparisons more stable.

In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# ---------------------------------------------------------
# 1. Check raw daily-table grain
# ---------------------------------------------------------

grain_duplicates = con.sql(f"""
    SELECT
        report_date,
        client_hash_id,
        content_hash_id,
        COUNT(*) AS n
    FROM {FACT_MAR}
    GROUP BY 1, 2, 3
    HAVING COUNT(*) > 1
    LIMIT 5
""").df()

print("Raw daily grain duplicate examples:")
display(grain_duplicates)

assert len(grain_duplicates) == 0
print("✓ Raw grain holds: one row per date + client + content.")


# ---------------------------------------------------------
# 2. Build March feature aggregates
# ---------------------------------------------------------

march = con.sql(f"""
    SELECT
        client_hash_id,
        content_hash_id,

        SUM(gsc_impressions) AS feature_impressions,
        SUM(gsc_clicks) AS feature_clicks,

        CASE
            WHEN SUM(gsc_impressions) > 0
            THEN 100.0 * SUM(gsc_clicks) / SUM(gsc_impressions)
        END AS feature_ctr,

        AVG(
            CASE
                WHEN gsc_impressions > 0
                THEN gsc_avg_position
            END
        ) AS feature_avg_position

    FROM {FACT_MAR}

    GROUP BY 1, 2
""").df()


# ---------------------------------------------------------
# 3. Build April outcome aggregates
# ---------------------------------------------------------

april = con.sql(f"""
    SELECT
        client_hash_id,
        content_hash_id,

        SUM(gsc_impressions) AS outcome_impressions,
        SUM(gsc_clicks) AS outcome_clicks,

        CASE
            WHEN SUM(gsc_impressions) > 0
            THEN 100.0 * SUM(gsc_clicks) / SUM(gsc_impressions)
        END AS outcome_ctr,

        AVG(
            CASE
                WHEN gsc_impressions > 0
                THEN gsc_avg_position
            END
        ) AS outcome_avg_position

    FROM {FACT_APR}

    GROUP BY 1, 2
""").df()


# ---------------------------------------------------------
# 4. Join feature and outcome windows
# ---------------------------------------------------------

contract_df = march.merge(
    april,
    on=["client_hash_id", "content_hash_id"],
    how="inner"
)

print(f"\nMarch content items: {len(march):,}")
print(f"April content items: {len(april):,}")
print(f"Items observed in both windows: {len(contract_df):,}")


# Check final grain
duplicates = contract_df.duplicated(
    ["client_hash_id", "content_hash_id"]
).sum()

print(f"Duplicate final-grain rows: {duplicates}")

assert duplicates == 0
print("✓ Final grain holds: one row per client + content.")


# ---------------------------------------------------------
# 5. Missing-value / usability checks
# ---------------------------------------------------------

check_columns = [
    "feature_impressions",
    "feature_clicks",
    "feature_ctr",
    "feature_avg_position",
    "outcome_impressions",
    "outcome_clicks",
    "outcome_ctr",
    "outcome_avg_position",
]

missing_report = (
    contract_df[check_columns]
    .isna()
    .mean()
    .mul(100)
    .round(2)
    .rename("missing_percent")
    .to_frame()
)

print("\nMissingness:")
display(missing_report)


# ---------------------------------------------------------
# 6. Create position-adjusted April opportunity proxy
# ---------------------------------------------------------

eligible = contract_df[
    (contract_df["feature_impressions"] >= 500)
    & (contract_df["outcome_impressions"] >= 500)
    & (contract_df["outcome_avg_position"].notna())
].copy()


def position_tier(pos):
    if pos <= 3:
        return "top_3"
    elif pos <= 10:
        return "page_1"
    elif pos <= 20:
        return "striking"
    elif pos <= 50:
        return "page_3_5"
    else:
        return "deep"


eligible["outcome_position_tier"] = (
    eligible["outcome_avg_position"].apply(position_tier)
)

eligible["outcome_tier_median_ctr"] = (
    eligible.groupby("outcome_position_tier")["outcome_ctr"]
    .transform("median")
)

eligible["ctr_gap_pp"] = (
    eligible["outcome_tier_median_ctr"]
    - eligible["outcome_ctr"]
)

eligible["opportunity_proxy"] = (
    eligible["ctr_gap_pp"] > 0.10
)

print(f"\nEligible modeling rows: {len(eligible):,}")

print(
    f"April CTR-opportunity proxy rows: "
    f"{eligible['opportunity_proxy'].sum():,} "
    f"({eligible['opportunity_proxy'].mean():.1%})"
)

display(
    eligible[
        [
            "feature_impressions",
            "feature_ctr",
            "feature_avg_position",
            "outcome_impressions",
            "outcome_ctr",
            "outcome_position_tier",
            "ctr_gap_pp",
            "opportunity_proxy",
        ]
    ].head()
)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Raw daily grain duplicate examples:


,report_date,client_hash_id,content_hash_id,n


✓ Raw grain holds: one row per date + client + content.


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


March content items: 331,437
April content items: 362,172
Items observed in both windows: 331,436
Duplicate final-grain rows: 0
✓ Final grain holds: one row per client + content.

Missingness:


,missing_percent
feature_impressions,0.00
feature_clicks,0.00
feature_ctr,46.68
feature_avg_position,46.68
outcome_impressions,0.00
outcome_clicks,0.00
outcome_ctr,46.76
outcome_avg_position,46.76



Eligible modeling rows: 51,496
April CTR-opportunity proxy rows: 11,342 (22.0%)


,feature_impressions,feature_ctr,feature_avg_position,outcome_impressions,outcome_ctr,outcome_position_tier,ctr_gap_pp,opportunity_proxy
1,602.0,0.664452,4.428747,879.0,0.113766,page_1,0.086104,False
4,1858.0,0.322928,1.854929,1587.0,0.063012,top_3,0.284452,True
8,10849.0,0.202784,8.240351,16121.0,0.142671,page_1,0.057199,False
16,2099.0,0.047642,3.066796,1472.0,0.271739,page_1,-0.071869,False
19,735.0,2.176871,3.173884,3681.0,1.140994,top_3,-0.793531,False


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*


This dataset supports observational and directional decision-support analysis, but it has important limits.

First, the warehouse is an **unbalanced panel**. Different clients have different history lengths, so a missing historical period cannot automatically be interpreted as zero traffic.

Second, some early rows contain search measurements before GA4 tracking became available. GA4 zeros from those periods must not be interpreted as zero engagement. Any future engagement extension of this contract must use the GA4 availability flag explicitly.

Third, time windows must remain separated. April outcome measurements cannot become March features. Query-level 90-day fields also require special care because their fixed 90-day window may overlap an outcome period and cause leakage.

Fourth, low-volume CTR measurements are noisy. A page with very few impressions can have a large CTR change because of only one or two clicks, so the analysis applies minimum-volume requirements.

Finally, these data are observational. A low CTR opportunity score cannot prove that changing a title, metadata, snippet, or content caused or will cause better performance. The output is a ranked human-review aid, not an automatic publishing decision.

In [7]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


# Verify some of the documented limitations.

client_coverage = con.sql(f"""
    SELECT
        COUNT(*) AS clients,
        MIN(gsc_data_start) AS earliest_gsc_start,
        MAX(gsc_data_start) AS latest_gsc_start,
        MIN(ga4_data_start) AS earliest_ga4_start,
        MAX(ga4_data_start) AS latest_ga4_start
    FROM {DIM_CLIENTS}
""").df()

display(client_coverage)

earliest_gsc = client_coverage.loc[0, "earliest_gsc_start"]
latest_gsc = client_coverage.loc[0, "latest_gsc_start"]

print(
    "Different GSC start dates:",
    earliest_gsc != latest_gsc
)

assert earliest_gsc != latest_gsc

print(
    "✓ Client histories do not all begin on the same date; "
    "the panel is unbalanced."
)

,clients,earliest_gsc_start,latest_gsc_start,earliest_ga4_start,latest_ga4_start
0,104,2025-01-27,2026-06-02,2025-10-29,2026-06-01


Different GSC start dates: True
✓ Client histories do not all begin on the same date; the panel is unbalanced.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.